In [ ]:
# =============================================================================
# explore/Explore_telemetry_mapping.py
# See what is going on with the telemetry mapping (job 16): which rows matched, which did
# not and why, and what rule would match more of them.
#
# RUN ALL CELLS. READ ONLY: it reads ea_rt_telemetry_mapping, ea_rt_measure_register and
# ea_rt_stations and writes nothing. Each section prints a count and shows a table (use the
# download arrow on a table to take it away as CSV).
#
#   1  Overview             how many rows in each status, and what share of measures are mapped
#   2  Matched              by what is measured, and the matches whose names disagree (check these)
#   3  no_station           rows whose TelemetryID is not a station reference (rows whose name says INACTIVE
#                           are already set aside as status inactive). A station that is listed but has no
#                           readings in the feed (Welsh sites) cannot be matched by any rule, so each route is
#                           split into reachable and not. Looks each one up in
#                           ea_rt_stations by RLOIid, notation, wiskiID (the file's WISKI_ID too), the
#                           ID tidied up, and finally a similar name, and counts what each route
#                           would find. This is where a new matching rule comes from.
#   4  ambiguous            why several measures fit, whether choosing the shortest period settles it, and
#                           how many are just a Stage and a Downstream Stage (and what the file says about them)
#   5  unknown_parameter    the ParameterID and TelemetryParameter words I do not map
#   6  no_parameter         stations that exist but lack that kind of measure
#   7  Measures with no mapping   register measures nothing points at, and why
#
# Widget: rows_to_show (how many lines of each detail table to display).
# Run notebook 16 first, and again after any change to the file or the rules, so this reads fresh rows.
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd

import config
from utils.helpers import table_exists
from utils.mapping_diagnostics import (
    station_candidates, candidate_summary, candidate_measures, tidal_summary, classify_ambiguity, stage_pairs,
    unmapped_measures, matched_name_check, profile_unplaced
)
from utils.telemetry_mapping import family_of_text

dbutils.widgets.text("rows_to_show", "200", "Rows to show in each detail table")
ROWS = int(dbutils.widgets.get("rows_to_show") or 200)

mapping_table = config.TBL_TELEMETRY_MAPPING
for t in (mapping_table, config.TBL_EA_RT_REGISTER):
    if not table_exists(spark, t):
        raise Exception(f"{t} does not exist. Run 16 first.")

rows = spark.table(mapping_table).toPandas()
_reg_cols = spark.table(config.TBL_EA_RT_REGISTER).columns
register = spark.table(config.TBL_EA_RT_REGISTER).select(
    *[c for c in ("measure_uri", "station_reference", "station_label", "parameter", "qualifier", "value_type", "period", "unit_name") if c in _reg_cols]).toPandas()
stations = spark.table(config.TBL_EA_RT_STATIONS).toPandas() if table_exists(spark, config.TBL_EA_RT_STATIONS) else pd.DataFrame()
if stations.empty:
    print("NOTE: ea_rt_stations does not exist (job 15), so section 3 can only use the register.")
    stations = pd.DataFrame({"stationReference": register["station_reference"].drop_duplicates(),
                             "label": register.drop_duplicates("station_reference")["station_label"]})
print(f"Mapping rows: {len(rows):,}   register measures: {len(register):,}   stations: {len(stations):,}")


In [ ]:
# =============================================================================
# 1. OVERVIEW
# =============================================================================

status = rows.groupby("match_status")["id"].nunique().sort_values(ascending=False).rename("mapping_rows").reset_index()
display(status)

matched_measures = rows.loc[rows["match_status"] == "matched", "measure_uri"].nunique()
print(f"Measures with a match: {matched_measures:,} of {len(register):,} ({100 * matched_measures / len(register):.0f}%)")
print(f"Mapping rows that matched: {int((rows.drop_duplicates('id')['match_status'] == 'matched').sum()):,} of {rows['id'].nunique():,}")
print("An ambiguous row appears once per candidate in the table but is counted once here.")


In [ ]:
# =============================================================================
# 2. MATCHED
# By what is measured, and the matched rows whose names disagree.
# =============================================================================

matched = rows[rows["match_status"] == "matched"]
by_family = matched.groupby(matched["telemetry_family"].fillna("(from ParameterID)"))["measure_uri"].nunique().rename("measures").reset_index()
display(by_family)

more_than_one = matched.groupby("measure_uri")["id"].nunique()
print(f"Measures matched by more than one row of the file: {int((more_than_one > 1).sum()):,} "
      f"(for example two FFFS ids for one gauge)")
display(matched[matched["measure_uri"].isin(more_than_one[more_than_one > 1].index)]
        .sort_values("measure_uri")[["measure_uri", "telemetry_id", "fffs_id", "fffs_location_name", "wiski_id"]].head(ROWS))

# The check that matters: a station matched on the ID whose name is nothing like the gauge's name.
# A matched measure that has never delivered a reading is mapped but dead (n_readings is filled by 16 when it settles by readings).
if "n_readings" in matched and matched["n_readings"].notna().any():
    nodata = matched[pd.to_numeric(matched["n_readings"], errors="coerce").fillna(0) == 0]
    print(f"\nMatched measures with NO readings at all: {nodata['measure_uri'].nunique():,} of {matched['measure_uri'].nunique():,}")
    display(nodata[["telemetry_id", "fffs_location_name", "station_label", "register_parameter", "register_qualifier", "period", "measure_uri"]].head(ROWS))
    settled = matched[matched["match_note"].fillna("").str.startswith("settled by readings")]
    print(f"Rows settled because only one candidate had readings: {settled['id'].nunique():,}")
    display(settled[["telemetry_id", "fffs_location_name", "register_qualifier", "period", "n_readings", "last_reading", "match_note"]].head(ROWS))

names = matched_name_check(rows, below=0.4)
print(f"\nMatched rows whose FFFS name and station label barely overlap (similarity under 0.4): {len(names):,}")
print("Some are fine (one is a site name, the other a road), but a wrong match shows up here first.")
display(names.head(ROWS))


In [ ]:
# =============================================================================
# 3. no_station: the TelemetryID is not a station reference in the register
# For each such row, look for the station by every other key the row or the station carries.
# =============================================================================

no_station = rows[rows["match_status"] == "no_station"]
print(f"no_station rows: {no_station['id'].nunique():,}")

cands = station_candidates(no_station, stations, register_refs=register["station_reference"].dropna().unique())
summary = candidate_summary(no_station, cands)
print("\nWhat each route would have found (each row counted once, at its best route):")
display(summary)
print("RLOIid, notation and wiskiID are exact keys the station carries: a safe rule. 'same after tidy-up' is a "
      "formatting difference (case, dashes, leading zeros). 'similar name' is a guess: check it by eye.")
print("'reachable' means the station is in the register, so a rule using that route would gain a match. "
      "'listed, no readings in the feed' means ea_rt_stations lists the station but the register has no measures for it "
      "(Welsh sites are the usual case), so there is nothing to link and it cannot be fixed by a rule.")
if "reachable" in summary and summary["reachable"].notna().any():
    gain = int(summary.loc[summary["reachable"] == "reachable", "rows"].sum())
    out_of_reach = int(summary.loc[summary["reachable"] == "listed, no readings in the feed", "rows"].sum())
    print(f"\nRows a new rule could match: {gain:,}.  Rows with a listed station but no readings: {out_of_reach:,}.")

display(cands.sort_values(["how", "score"], ascending=[True, False]).head(ROWS))

# Rows no route found: what do they look like?
unfound = no_station[~no_station["id"].isin(cands["id"])]
print(f"\nNo candidate by any route: {unfound['id'].nunique():,}. By what they measure:")
display(unfound.drop_duplicates("id").groupby(["telemetry_parameter"]).size().rename("rows").sort_values(ascending=False).reset_index().head(30))
display(unfound.drop_duplicates("id")[["telemetry_id", "telemetry_parameter", "parameter_id", "fffs_location_name", "wiski_id"]].head(ROWS))

# What the rows no route could place have in common, so a rule or an out-of-scope list can be made for them.
# Welsh sites are the usual case: the real-time feed does not carry them. The name hint is only a hint.
prof = profile_unplaced(unfound)
print("\nWhat the unplaced rows have in common:")
display(prof["hints"])
print("Shape of the TelemetryID (9 is a digit, A a letter): a Welsh scheme may show up as one shape.")
display(prof["shapes"].head(ROWS))
print("First two characters of the TelemetryID:")
display(prof["prefixes"].head(ROWS))
print("What they measure:")
display(prof["parameters"].head(ROWS))
print("To set them aside, give notebook 16 a CSV of them (FFFS_ID and/or TelemetryID, and a Reason) in its out_of_scope_csv widget. "
      "If the Incident Management Reference Database can export the locations with their area or region, filter that to Wales.")

# =============================================================================
# 3b. WHAT THE SIMILAR-NAME CANDIDATES ARE
# A name match is a guess, and a place often has several stations (Immingham has three). This lists the measures
# behind each candidate, with the value type (instantaneous, mean ...) and whether it is a tidal level, so the
# ambiguity can be seen before any rule is chosen.
# =============================================================================

named = cands[cands["how"] == "similar name"]
cm = candidate_measures(named, register)
ts = tidal_summary(cm)
print(f"Rows reached only by a similar name: {named['id'].nunique():,}")
if len(ts):
    print(f"  with at least one tidal-level candidate: {int(ts['any_tidal'].sum()):,}")
    print(f"  reaching more than one station: {int((ts['stations'] > 1).sum()):,}   reaching more than one measure: {int((ts['measures'] > 1).sum()):,}")
    display(ts.groupby("value_types").size().rename("rows").sort_values(ascending=False).reset_index().head(ROWS))
    display(cm.sort_values(["telemetry_id", "candidate_station_reference"]).head(ROWS))


In [ ]:
# =============================================================================
# 4. ambiguous: more than one measure fits
# =============================================================================

amb = classify_ambiguity(rows)
print(f"ambiguous mapping rows: {len(amb):,}")
display(amb.groupby("why").agg(rows=("id", "count"), settled_by_shortest_period=("settled_by_shortest_period", "sum")).reset_index())
print("settled_by_shortest_period: choosing the measure with the shortest period (usually the 15-minute one) leaves exactly one.")
display(amb.head(ROWS))

# What each remaining candidate has by way of readings: if both deliver data they are genuinely both live.
still = rows[rows["match_status"] == "ambiguous"]
if len(still) and "n_readings" in still:
    print("\nThe remaining candidates and their readings (a rule that picks the one with data has already been applied):")
    display(still.sort_values(["telemetry_id", "measure_uri"])[["telemetry_id", "fffs_location_name", "register_qualifier", "period", "n_readings", "last_reading", "measure_uri"]].head(ROWS))

# Is it a Stage and a Downstream Stage? They share the station, so they share the TelemetryID, and
# probably the WISKI and FFFS ids. This shows what the file says about each such station.
sp = stage_pairs(rows)
print(f"\nAmbiguous rows that are just a Stage and a Downstream Stage: {len(sp):,} stations")
if len(sp):
    display(sp.groupby("file_describes").size().rename("stations").reset_index())
    print("one_row_no_qualifier: the file has one row and no qualifier, so it does not say which measure it means.\n"
          "distinct_qualifiers:  the file has a row for each; the qualifier should have settled it, so look for a spelling difference.\n"
          "same_qualifier:       several rows, one qualifier.   other: something else.")
    display(sp.head(ROWS))
    shared = int(((sp["wiski_ids"].str.count(r"\|") == 0) & (sp["wiski_ids"] != "")).sum())
    print(f"Stations where every file row carries the same single WISKI id: {shared:,} of {len(sp):,}")


In [ ]:
# =============================================================================
# 5. unknown_parameter: neither column names a parameter the code knows
# =============================================================================

unknown = rows[rows["match_status"] == "unknown_parameter"].drop_duplicates("id")
print(f"unknown_parameter rows: {len(unknown):,}")
display(unknown.groupby(["parameter_id", "telemetry_parameter"], dropna=False).size().rename("rows").sort_values(ascending=False).reset_index().head(ROWS))
print("Each pair above is a word the code does not map to level, flow, rainfall, groundwater, wave or shingle. "
      "Tell me what each means and it is a one-line addition.")

# =============================================================================
# 6. no_parameter: the station exists but has no measure of that kind
# =============================================================================

npar = rows[rows["match_status"] == "no_parameter"].drop_duplicates("id")
print(f"\nno_parameter rows: {len(npar):,}")
display(npar[["telemetry_id", "telemetry_parameter", "parameter_id", "fffs_location_name", "match_note"]].head(ROWS))


In [ ]:
# =============================================================================
# 7. MEASURES WITH NO MAPPING
# Register measures that no matched row points at: the other side of the same gap.
# =============================================================================

um = unmapped_measures(register, rows)
print(f"Register measures with no mapping: {len(um):,} of {len(register):,}")
display(um.groupby(["why", "family"]).size().rename("measures").reset_index().sort_values("measures", ascending=False))
print("'station is in the file, this parameter is not matched': the file knows the gauge but a different parameter, "
      "or the row did not match. These are the ones a better rule could reach.")
display(um[um["why"].str.startswith("station is in the file")].head(ROWS))
